# Practice: Building and Training a CNN with PyTorch

Use this practice after Section 6.

- Download and extract the digit images.
- Load the digit images and inspect the image tensors.
- Predict batch shapes and verify what happens in the final batch.
- Trace a batch through convolution, pooling and classification.
- Train a CNN and inspect errors on held-out images.

Run the notebook from top to bottom in a fresh runtime. A GPU is optional.

Use Colab's standard Python runtime. For local Jupyter, make sure `sample_data/` exists in the working directory, `curl` and `unzip` are available, and the required packages are installed (`python -m pip install torch torchvision scikit-learn`).

[Open in Colab](https://colab.research.google.com/github/wlos/ai_with_python_course/blob/main/16-Deep-Learning-and-Image-Processing/practice/Practice%20CNN%20Training%20with%20PyTorch.ipynb)


## 1. Download and extract the digit images

The `!` prefix runs a terminal command from a notebook cell. The cell downloads `digit_images.zip` into Colab's `sample_data/` folder and extracts it to `sample_data/digit_images/`. The `-n` option skips files that already exist.


In [ ]:
!curl -fLsS -o sample_data/digit_images.zip https://raw.githubusercontent.com/wlos/AI_SS/main/13-Computer-Vision/sample_data/digit_images.zip
!unzip -q -n sample_data/digit_images.zip


## 2. Inspect image transforms

The PNGs are grouped in folders named `0` to `9`. Before running the cell, predict which transform sets the number of channels and which changes the pixel value range. Check the class mapping and explain why the first label is zero.


In [ ]:
from torchvision import datasets, transforms

DIGIT_ROOT = "sample_data/digit_images"

digit_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),  # Keep a single intensity channel.
    transforms.Resize((28, 28)),
    transforms.ToTensor(),  # Convert pixels to float values in [0, 1].
])
digit_dataset = datasets.ImageFolder(DIGIT_ROOT, transform=digit_transform)
sample_image, sample_label = digit_dataset[0]
print(f"Classes: {digit_dataset.class_to_idx}")
print(f"Image shape: {tuple(sample_image.shape)} | Label: {sample_label}")
assert len(digit_dataset) == 240 and len(digit_dataset.classes) == 10
assert tuple(sample_image.shape) == (1, 28, 28)


## 3. Prepare image batches

The seeded `random_split` gives 192 training images and 48 test images. Change `batch_size=16` to `batch_size=20` in both loaders. Predict the first and last batch sizes for each split. Check your predictions with the printed results and explain why the test loader keeps the image order fixed.


In [ ]:
import torch
from torch.utils.data import DataLoader, random_split

DIGIT_SEED = 42

# Randomly assign each image to exactly one subset using a fixed seed.
train_size = int(0.8 * len(digit_dataset))
test_size = len(digit_dataset) - train_size
digit_train, digit_test = random_split(
    digit_dataset,
    [train_size, test_size],
    generator=torch.Generator().manual_seed(DIGIT_SEED),
)
digit_train_loader = DataLoader(
    digit_train, batch_size=16, shuffle=True,
    generator=torch.Generator().manual_seed(DIGIT_SEED),
)
digit_test_loader = DataLoader(digit_test, batch_size=16, shuffle=False)

batch_images, batch_labels = next(iter(digit_test_loader))
print(f"Training images: {len(digit_train)} | Test images: {len(digit_test)}")
print(f"Image batch: {tuple(batch_images.shape)} | Label batch: {tuple(batch_labels.shape)}")
print(f"Batch labels: {batch_labels.tolist()}")

# Inspect every batch size, including the final batch.
print("Training batch sizes:", [len(labels) for _, labels in digit_train_loader])
print("Test batch sizes:", [len(labels) for _, labels in digit_test_loader])


## 4. Build the two-block CNN

Read the two convolution–ReLU–pooling blocks. For each block, write down how many channels the convolution produces and how pooling changes the height and width. Use your answers to explain why the linear layer takes `16 * 7 * 7` values per image.


In [ ]:
from torch import nn

DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

class DigitCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 8, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),  # Reduce 28 × 28 to 14 × 14.
            nn.Conv2d(8, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),  # Reduce 14 × 14 to 7 × 7.
        )
        self.flatten = nn.Flatten()  # Keep the batch dimension separate.
        self.head = nn.Linear(16 * 7 * 7, num_classes)

    def forward(self, images):
        features = self.features(images)
        return self.head(self.flatten(features))

torch.manual_seed(DIGIT_SEED)  # Reproduce the model's initial weights.
digit_class_count = len(digit_dataset.classes)
digit_cnn = DigitCNN(digit_class_count).to(DEVICE)


## 5. Train the CNN

For each training batch, identify the line that clears gradients, the line that calculates gradients and the line that updates the weights. Record the training loss after the first and last epochs.


In [ ]:
DIGIT_EPOCHS = 3
DIGIT_LEARNING_RATE = 0.001
digit_loss = nn.CrossEntropyLoss()

def train_digit_epoch(model, loader, optimiser):
    '''Update trainable parameters and return mean loss per image.'''
    total_loss = 0.0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimiser.zero_grad()  # Clear gradients left by the previous batch.
        loss = digit_loss(model(images), labels)
        loss.backward()  # Calculate gradients from the loss.
        optimiser.step()  # Apply an update to the optimiser's parameters.
        total_loss += loss.item() * len(labels)  # Weight each batch by its size.
    return total_loss / len(loader.dataset)

digit_optimiser = torch.optim.Adam(digit_cnn.parameters(), lr=DIGIT_LEARNING_RATE)
digit_cnn.train()
for epoch in range(DIGIT_EPOCHS):
    mean_loss = train_digit_epoch(digit_cnn, digit_train_loader, digit_optimiser)
    print(f"Simple CNN epoch {epoch + 1}: loss={mean_loss:.3f}")


## 6. Test and evaluate the CNN

Run the trained model on all 48 test images and collect the true and predicted digits. Then read the accuracy and per-digit statistics in the scikit-learn report. Find one incorrect prediction in the first batch and record its true and predicted digits. Explain why you would use a validation split to choose a different epoch count or learning rate.


In [ ]:
def predict_digits(model, loader):
    """Return true and predicted class indices in test-loader order."""
    model.eval()
    true_labels = []
    predictions = []
    with torch.no_grad():  # Inference does not need gradients.
        for images, labels in loader:
            logits = model(images.to(DEVICE))
            true_labels.extend(labels.tolist())
            predictions.extend(logits.argmax(dim=1).cpu().tolist())
    return true_labels, predictions

cnn_test_labels, cnn_test_predictions = predict_digits(digit_cnn, digit_test_loader)
print(f"Test predictions: {len(cnn_test_predictions)}")


In [ ]:
from sklearn.metrics import accuracy_score, classification_report

simple_digit_accuracy = accuracy_score(cnn_test_labels, cnn_test_predictions)
print(f"Simple CNN test accuracy: {simple_digit_accuracy:.1%}")
print(classification_report(
    cnn_test_labels,
    cnn_test_predictions,
    labels=list(range(len(digit_dataset.classes))),
    target_names=digit_dataset.classes,
    zero_division=0,
))

# Inspect the predictions already collected from the first test batch.
print("True digits:     ", cnn_test_labels[:len(batch_labels)])
print("Predicted digits:", cnn_test_predictions[:len(batch_labels)])
